In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
import time
import psutil
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
import numpy as np

In [2]:
# Check data hadoop
PATH_TO_CSV = "../../hdfs/data-input/drug200.parquet"
pass_linux = 'echo 123 | sudo -S'
!{pass_linux} docker exec namenode hdfs dfs -mkdir -p /user/data/drug
!{pass_linux} docker cp "{PATH_TO_CSV}" namenode:/tmp/drug200.parquet
!{pass_linux} docker exec namenode hdfs dfs -put /tmp/drug200.parquet /user/data/drug
!{pass_linux} docker exec namenode hdfs dfs -ls -R /user/data/drug

[sudo] password for lenovo: [sudo] password for lenovo: Successfully copied 395MB to namenode:/tmp/drug200.parquet
[sudo] password for lenovo: put: `/user/data/drug/drug200.parquet': File exists
[sudo] password for lenovo: -rw-r--r--   2 root supergroup  395495382 2026-05-06 12:15 /user/data/drug/drug200.parquet


In [3]:
memory_use = "20g"
spark = (SparkSession.builder
    .appName("Optimized_RF_50M_Rows")
    # Allocate 12GB for drivers, leaving 4GB for the OS and Docker overhead.
    .config("spark.driver.memory", memory_use)
    .config("spark.executor.memory", memory_use)
    
    # Memory Optimization: Allocate 80% of RAM to Spark,
    # Prioritizing Execution (RF computation) over Storage (cache)
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    
    # Data partitioning: 50 million rows should be divided into at least 500 partitions.
    .config("spark.sql.shuffle.partitions", "500")
    .config("spark.default.parallelism", "500")
    
    # Avoid timeouts when processing large volumes.
    .config("spark.network.timeout", "1200s")
    .config("spark.sql.broadcastTimeout", "1200s")

    .config("spark.jars.packages", "com.redislabs:spark-redis_2.12:3.1.0")

    .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/07 12:59:58 WARN Utils: Your hostname, lenovo-Legion-5-15IAH7H, resolves to a loopback address: 127.0.1.1; using 192.168.1.8 instead (on interface wlp0s20f3)
26/05/07 12:59:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/media/lenovo/D1/Data%20legion%202026/NTT_Exercises/Parallelcomputing_Distributedsystems/source-code/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/lenovo/.ivy2.5.2/cache
The jars for the packages stored in: /home/lenovo/.ivy2.5.2/jars
com.redislabs#spark-redis_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4cd6818b-bd2e-4bea-a1f3-138752214efd;1.0
	confs: [default]
	found com.redislabs#spark-redis_2.12;3.1.0 in central
	found redis.clients#jedis;3.9.0 in central
	found org.slf4j#slf4j-

In [ ]:
# Read from HDFS
ip_namenode = '172.20.0.2'
spark.sparkContext.setCheckpointDir(f"hdfs://{ip_namenode}:9000/user/checkpoints")
hdfs_path = f"hdfs://{ip_namenode}:9000/user/data/drug/drug200.parquet"
df = spark.read.parquet(hdfs_path, header=True, inferSchema=True)
# Check load data
df = df.repartition(500) 
df.show(5)
df.printSchema()

In [ ]:
df.count()

50000000

In [ ]:
# Lấy số lượng hàng (Row count)
row_count = df.count()
col_count = len(df.columns)
print(f"Kích thước tập dữ liệu: {row_count} hàng x {col_count} cột")

Kích thước tập dữ liệu: 50000000 hàng x 7 cột


In [ ]:
df['Age','Na_to_K'].describe().show()

+-------+------------------+------------------+
|summary|               Age|           Na_to_K|
+-------+------------------+------------------+
|  count|          50000000|          50000000|
|   mean|       44.50098632|20.499214869852036|
| stddev|17.318983621047654| 8.370598377268237|
|    min|                15|               6.0|
|    max|                74|              35.0|
+-------+------------------+------------------+



In [ ]:
# Đăng ký View tạm thời
df.createOrReplaceTempView("drug_filter")
# Truy vấn bằng SQL
df_filtered = spark.sql("""
    SELECT * 
    FROM drug_table 
    WHERE row_id >= 1100 AND row_id <= 1110
""")

df_filtered.show()

+------+---+---+------+-----------+-------+-----+
|row_id|Age|Sex|    BP|Cholesterol|Na_to_K| Drug|
+------+---+---+------+-----------+-------+-----+
|  1109| 70|  F|   LOW|     NORMAL|  8.797|DrugC|
|  1102| 28|  M|NORMAL|       HIGH|  9.337|DrugX|
|  1103| 51|  M|NORMAL|       HIGH| 33.701|DrugY|
|  1101| 68|  M|NORMAL|     NORMAL|  10.39|DrugX|
|  1105| 67|  F|NORMAL|       HIGH| 29.684|DrugY|
|  1104| 63|  F|NORMAL|     NORMAL| 17.601|DrugY|
|  1100| 50|  M|NORMAL|       HIGH| 22.656|DrugY|
|  1107| 46|  M|   LOW|       HIGH|  6.995|DrugC|
|  1110| 51|  F|  HIGH|       HIGH| 24.617|DrugY|
|  1108| 19|  F|NORMAL|     NORMAL| 22.332|DrugY|
|  1106| 29|  M|  HIGH|     NORMAL|  15.17|DrugY|
+------+---+---+------+-----------+-------+-----+



In [ ]:
from pyspark.sql.functions import col
result = df.filter(col("row_id") == 1500)